# 01 — Data Preprocessing

**NARR CAPE/CIN/Wind Shear → Cleaned Zarr Archive**

This notebook covers the first stage of the NARR-based Self-Organizing Map (SOM) pipeline:

1. **Loading** individual annual NetCDF files for CAPE, CIN, and vertical wind shear downloaded from NOAA PSL
2. **Merging** them into a single multi-year `xarray` dataset
3. **Flattening & cleaning** the spatial fields into a 2-D feature matrix for SOM input
4. **Saving** the merged dataset as a Zarr archive for efficient access in subsequent notebooks

---

### Data Source
NARR (North American Regional Reanalysis) data can be downloaded from the NOAA Physical Sciences Laboratory:
> https://psl.noaa.gov/data/gridded/data.narr.html

Download daily surface files for:
- `cape.YYYY.nc` — Convective Available Potential Energy (J/kg)
- `cin.YYYY.nc` — Convective Inhibition (J/kg)
- `vwsh.YYYY.nc` — Vertical Wind Shear (converted to knots)

Place all `.nc` files in the same directory before running this notebook.

### Output
- `cape_cin_NARR.zarr/` — merged, chunked Zarr store used by notebooks 02 and 03

## 1. Imports

In [ ]:
import xarray as xr
import numpy as np
import os

## 2. Configuration

Set the paths to your downloaded NetCDF files and the desired output location for the Zarr store.
Adjust `DATA_DIR`, `OUTPUT_DIR`, and `YEARS` to match your environment.

In [ ]:
# ── Paths ────────────────────────────────────────────────────────────────────
DATA_DIR   = "/path/to/narr_nc_files"          # Directory containing .nc files
OUTPUT_DIR = "/path/to/output"                  # Where the .zarr store will be written

ZARR_PATH  = os.path.join(OUTPUT_DIR, "cape_cin_NARR.zarr")

# ── Years to process ─────────────────────────────────────────────────────────
YEARS = [2018, 2019, 2020, 2021]

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output will be written to: {ZARR_PATH}")

## 3. Load and Merge Annual NetCDF Files

Each variable (CAPE, CIN, wind shear) is stored in separate per-year `.nc` files.
We open each file and then merge all years and variables into one dataset.

> **Note:** `xr.open_mfdataset` with `combine='by_coords'` is an alternative if your
> files follow a consistent naming convention, but the explicit loop below is easier
> to inspect and debug.

In [ ]:
cape_datasets = []
cin_datasets  = []
shr_datasets  = []

for year in YEARS:
    cape_path = os.path.join(DATA_DIR, f"cape.{year}.nc")
    cin_path  = os.path.join(DATA_DIR, f"cin.{year}.nc")
    shr_path  = os.path.join(DATA_DIR, f"vwsh.{year}.nc")  # vertical wind shear

    cape_datasets.append(xr.open_dataset(cape_path))
    cin_datasets.append(xr.open_dataset(cin_path))
    shr_datasets.append(xr.open_dataset(shr_path))
    print(f"Opened {year}")

# Concatenate along time, then merge all variables into one dataset
cape_merged = xr.concat(cape_datasets, dim="time")
cin_merged  = xr.concat(cin_datasets,  dim="time")
shr_merged  = xr.concat(shr_datasets,  dim="time")

Data = xr.merge([cape_merged, cin_merged, shr_merged])

print("\nMerged dataset summary:")
print(Data)

## 4. Inspect the Dataset

Before any processing, verify the dataset dimensions, coordinate ranges, and that all
expected variables are present.  The NARR grid is ~277 × 349 on a Lambert Conformal
projection; time should span the years in `YEARS` at daily frequency.

In [ ]:
print("Variables:", list(Data.data_vars))
print("Dimensions:", dict(Data.sizes))
print("Time range:", Data.time.values[0], "→", Data.time.values[-1])

for var in ["cape", "cin"]:
    if var in Data:
        arr = Data[var].values
        print(f"\n{var.upper()}:")
        print(f"  shape : {arr.shape}")
        print(f"  range : [{np.nanmin(arr):.1f}, {np.nanmax(arr):.1f}]")
        print(f"  NaNs  : {np.isnan(arr).sum()}")

## 5. Build the SOM Feature Matrix

The SOM expects a 2-D array of shape `(n_timesteps, n_features)`.  Each row is one
day; the columns are the flattened spatial grid cells of all input variables
concatenated side-by-side.

Cleaning steps applied:
- Replace non-finite values (NaN, ±Inf, fill values >1e20) with NaN
- Drop grid columns that are all-NaN across time (e.g. ocean/masked cells)
- Fill any remaining NaNs with the column mean (rare edge cells)
- Drop zero-variance columns (constants add no information to the SOM)

> The cleaned matrix `X` is **not** saved here — it is rebuilt in notebook 02 from the
> Zarr store so that scaling and SOM training remain reproducible in one place.

In [ ]:
n_time = Data.sizes["time"]

# Flatten spatial dimensions: (time, y, x) → (time, y*x)
X_cape = Data["cape"].values.reshape(n_time, -1).astype(np.float32)
X_cin  = Data["cin"].values.reshape(n_time, -1).astype(np.float32)

# Include wind shear if present
if "vwsh_kts" in Data:
    X_shr = Data["vwsh_kts"].values.reshape(n_time, -1).astype(np.float32)
    X_raw = np.concatenate([X_cape, X_cin, X_shr], axis=1)
else:
    X_raw = np.concatenate([X_cape, X_cin], axis=1)

print(f"Raw feature matrix shape: {X_raw.shape}")

# ── Clean non-finite values ───────────────────────────────────────────────────
X_raw[~np.isfinite(X_raw)] = np.nan
X_raw[np.abs(X_raw) > 1e20] = np.nan

# Drop columns that are entirely NaN (masked / ocean cells)
good_cols = ~np.isnan(X_raw).all(axis=0)
X_clean = X_raw[:, good_cols]
print(f"After all-NaN column drop: {X_clean.shape}")

# Fill remaining NaNs with the column mean
col_mean = np.nanmean(X_clean, axis=0)
col_mean = np.nan_to_num(col_mean, nan=0.0)
nan_idx = np.where(np.isnan(X_clean))
X_clean[nan_idx] = col_mean[nan_idx[1]]

# Drop zero-variance columns
std = np.std(X_clean, axis=0)
X_clean = X_clean[:, std > 0]

print(f"Final clean feature matrix: {X_clean.shape}")
print(f"NaNs remaining: {np.isnan(X_clean).sum()}")
print(f"All finite: {np.isfinite(X_clean).all()}")

## 6. Save Merged Dataset as Zarr

The merged xarray dataset (with lat/lon coordinates intact) is written to a Zarr store.
Zarr supports chunked, compressed storage and is much faster to open than multiple
individual NetCDF files.

Notebooks 02 and 03 will open this store directly with `xr.open_zarr()`.

In [ ]:
# Choose chunk size along time; spatial chunks follow the native NARR tile
chunk_config = {"time": 365, "y": 139, "x": 349}

Data_chunked = Data.chunk(chunk_config)

print(f"Writing Zarr store to: {ZARR_PATH}")
Data_chunked.to_zarr(ZARR_PATH, mode="w")
print("Done.")

## 7. Verify the Zarr Store

Re-open the store and confirm that variables, dimensions, and time range look correct
before moving on to notebook 02.

In [ ]:
ds_check = xr.open_zarr(ZARR_PATH)

print("Zarr store verified:")
print(ds_check)

print("\nVariables:", list(ds_check.data_vars))
print("Time range:", ds_check.time.values[0], "→", ds_check.time.values[-1])
print("Grid size (y × x):", ds_check.sizes.get("y"), "×", ds_check.sizes.get("x"))

---
## Next Steps

The Zarr store at `ZARR_PATH` is the input to **02_som_training.ipynb**, which handles:
- Feature scaling (MinMaxScaler)
- SOM configuration and training
- Best-matching unit (BMU) assignment
- Saving the trained SOM and BMU array as a pickle file